In [ ]:
!pip install earthengine-api geemap rasterio scikit-image

In [ ]:
import ee

ee.Authenticate()

ee.Initialize(project='YOUR_EE_PROJECT_ID') # Replace 'your-project-id' with your actual Google Cloud Project ID

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import os

BASE = "/content/drive/MyDrive/PyroRL_Saudi_Project/datasets"

folders = [
    "saudi_eastern_province/raw/ndvi/vegetation",

    "saudi_eastern_province/processed/fuel",

    "saudi_eastern_province/grids/32x32",

    "saudi_eastern_province/grids/64x64"
]

for folder in folders:
    os.makedirs(os.path.join(BASE, folder), exist_ok=True)

print("NDVI folders created.")

In [ ]:
saudi_roi = ee.Geometry.Rectangle([
    45.5,
    23.5,
    50.5,
    28.5
])

In [ ]:
modis = ee.ImageCollection("MODIS/061/MOD13Q1")

In [ ]:
ndvi_collection = (
    modis
    .filterDate('2025-06-01', '2025-06-30')
    .select('NDVI')
)

In [ ]:
ndvi = ndvi_collection.mean()

In [ ]:
ndvi_saudi = ndvi.clip(saudi_roi)

In [ ]:
import geemap

Map = geemap.Map(center=[26.5, 49], zoom=6)

ndvi_vis = {
    'min': 0,
    'max': 9000,
    'palette': [
        'white',
        'yellow',
        'green',
        'darkgreen'
    ]
}

Map.addLayer(ndvi_saudi, ndvi_vis, "Saudi NDVI")

Map

In [ ]:
import geemap

output_tif = "/content/drive/MyDrive/PyroRL_Saudi_Project/datasets/saudi_eastern_province/raw/ndvi/vegetation/saudi_ndvi.tif"

geemap.ee_export_image(
    ndvi_saudi,
    filename=output_tif,
    scale=250,
    region=saudi_roi,
    file_per_band=False
)

print("NDVI exported.")

In [ ]:
import os

print(os.path.exists(output_tif))

In [ ]:
import rasterio

src = rasterio.open(output_tif)

ndvi_data = src.read(1)

print(ndvi_data.shape)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,8))

plt.imshow(ndvi_data, cmap='YlGn')

plt.title("Raw Saudi NDVI")

plt.colorbar()

plt.show()

In [ ]:
import numpy as np

ndvi_data = ndvi_data.astype(np.float32)

ndvi_data = np.nan_to_num(ndvi_data)

In [ ]:
ndvi_scaled = ndvi_data * 0.0001

In [ ]:
print("Min:", ndvi_scaled.min())

print("Max:", ndvi_scaled.max())

print("Mean:", ndvi_scaled.mean())

In [ ]:
fuel_map = (ndvi_scaled + 1) / 2

In [ ]:
fuel_map = np.clip(fuel_map, 0, 1)

In [ ]:
plt.figure(figsize=(10,8))

plt.imshow(fuel_map, cmap='YlGn')

plt.title("Fuel Density Map")

plt.colorbar(label="Fuel Density")

plt.show()

In [ ]:
!pip install scikit-image

In [ ]:
from skimage.transform import resize

In [ ]:
fuel_32 = resize(
    fuel_map,
    (32,32),
    anti_aliasing=False
)

fuel_64 = resize(
    fuel_map,
    (64,64),
    anti_aliasing=False
)

In [ ]:
print(fuel_32.shape)

print(fuel_64.shape)

In [ ]:
plt.figure(figsize=(6,6))

plt.imshow(fuel_32, cmap='YlGn')

plt.title("32x32 Fuel Grid")

plt.colorbar()

plt.show()

In [ ]:
GRID_BASE = "/content/drive/MyDrive/PyroRL_Saudi_Project/datasets/saudi_eastern_province/grids"

np.save(
    f"{GRID_BASE}/32x32/fuel.npy",
    fuel_32
)

np.save(
    f"{GRID_BASE}/64x64/fuel.npy",
    fuel_64
)

print("Fuel grids saved.")

In [ ]:
import os

print(os.listdir(f"{GRID_BASE}/32x32"))

In [ ]:
fuel_test = np.load(f"{GRID_BASE}/32x32/fuel.npy")

print(fuel_test.shape)

print(fuel_test.min())

print(fuel_test.max())

In [ ]:
save_path = "/content/drive/MyDrive/PyroRL_Saudi_Project/visualizations/fuel_32.png"

plt.figure(figsize=(6,6))

plt.imshow(fuel_32, cmap='YlGn')

plt.title("32x32 Fuel Grid")

plt.colorbar()

plt.savefig(save_path, dpi=300)

plt.show()

print("Visualization saved.")

In [ ]:
print("NDVI Min:", ndvi_scaled.min())

print("NDVI Max:", ndvi_scaled.max())

print("NDVI Mean:", ndvi_scaled.mean())

print("Negative Values:", (ndvi_scaled < 0).sum())

In [ ]:
fuel_map = np.clip(ndvi_scaled, 0, 1)

In [ ]:
plt.figure(figsize=(10,8))

plt.hist(fuel_map.flatten(), bins=50)

plt.title("Fuel Distribution")

plt.xlabel("Fuel Density")

plt.ylabel("Frequency")

plt.show()

In [ ]:
sparse_fuel_mask = fuel_map < 0.15

moderate_fuel_mask = (fuel_map >= 0.15) & (fuel_map < 0.4)

dense_fuel_mask = fuel_map >= 0.4

In [ ]:
import json

metadata = {
    "dataset": "MODIS MOD13Q1",
    "region": "Saudi Eastern Province",
    "source_resolution_m": 250,
    "grid_sizes": [32, 64],
    "representation": "fuel density",
    "fuel_conversion": "clip_ndvi_0_1"
}

metadata_path = f"{GRID_BASE}/fuel_metadata.json"

with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=4)

print("Fuel metadata saved.")